In [37]:
using Revise
using NonlinearNormalForm, BeamTracking, Beamlines, GTPSA
import NonlinearNormalForm as NNF
using JET, BenchmarkTools
using SciBmad

Precompiling packages...
    345.7 ms  ✓ Accessors → TestExt
  1 dependency successfully precompiled in 1 seconds. 11 already precompiled.
Precompiling packages...
    342.3 ms  ✓ RecursiveArrayTools → RecursiveArrayToolsStatisticsExt
  1 dependency successfully precompiled in 0 seconds. 30 already precompiled.


In [29]:
include("../SciBmad/lattices/esr-v6.3.1.jl");
# Set RF for 0.05 synch tune (from previous optimization)
cavities = ring.line[findall(x->x.kind=="RFCavity", ring.line)];
foreach(x->x.voltage=3.355606790694894e6, cavities);

# Set all elements radiation damping, second order Yoshida
foreach(x->x.tracking_method=Yoshida(order=2, radiation_damping_on=true), ring.line);

t = twiss(ring)
co = [t.orbit_x[1], t.orbit_px[1], t.orbit_y[1], t.orbit_py[1], t.orbit_z[1], t.orbit_pz[1]]
# Closed orbit from previous check:
#co = [-2.6000686032132613e-7, 1.5303928131191435e-5, -3.556902826710349e-9, 3.9231617960303014e-8, -0.02704383542507716, -0.0003522717640595932];

6-element Vector{Float64}:
 -2.6000686032132613e-7
  1.5303928131191435e-5
 -3.556902826710349e-9
  3.9231617960303014e-8
 -0.02704383542507716
 -0.0003522717640595932

In [30]:
d4 = Descriptor(6, 4);
b0 = Bunch(co+vars(d4));
track!(b0, ring);
m = DAMap(v0=co, v=b0.coords.v)
norm(scalar.(m.v) .- co) < 1e-12 || println("not on closed orbit")

Setting bunch.species = Species(electron, charge=-1.0e, mass=510998.95069 eV/c², spin=0.5ħ) (reference species from the Beamline)
Setting bunch.R_ref = -59.5287244902766 (reference R_ref from the Beamline)


true

In [32]:
# Check symplectic error:
a = normal(m)
a0, a1, a2 = factorize(a)
ac = a ∘ canonize(a1, damping=true);
a0, a1, a2 = factorize(ac)
println(compute_sagan_rubin(a1, Val{true}()).beta[1])
ba = Bunch(ac.v[1:6])
track!(ba, ring)
a_end = DAMap(v=ba.coords.v)
damp = [0.,0.,0.]
ac_end = a_end∘canonize(a_end, damping=true, damp=damp)
a0end, a1end, a2end = factorize(ac_end)
println(compute_sagan_rubin(a1end, Val{true}()).beta[1])

0.590660168880395
Setting bunch.species = Species(electron, charge=-1.0e, mass=510998.95069 eV/c², spin=0.5ħ) (reference species from the Beamline)
Setting bunch.R_ref = -59.5287244902766 (reference R_ref from the Beamline)
0.5906601688803629


In [43]:
damp

3-element Vector{Float64}:
 0.0010461401064894994
 0.0010538510381837948
 0.002115411357111029

In [47]:
c = c_map(a)
r = inv(c)*inv(a)*m*a*c
damp+[real(log(r.v[2][2])), real(log(r.v[4][4])), real(log(r.v[6][6]))]

3-element Vector{Float64}:
 -4.8693687970668975e-15
  4.908540674614104e-10
 -2.454139677934841e-10

In [42]:
@report_opt canonize(a1,damping=true,damp=damp)

No errors detected


In [36]:
NNF.getscalar(a0∘a1∘a2)-co

6-element StaticArraysCore.SVector{6, Float64} with indices SOneTo(6):
 -2.093187819254827e-16
  1.3822561259653476e-16
  5.995405079784345e-20
  1.7336713022376396e-18
  1.056932319443149e-13
  2.770732861895331e-15

In [19]:
a_end2 = DAMap(v0=co,v=ba.coords.v)
scalar.(a_end2.v) - co

6-element StaticArraysCore.MVector{6, Float64} with indices SOneTo(6):
 -6.169199653776737e-6
 -1.4400708388299533e-5
  8.68066056709423e-9
 -1.0499466453552429e-7
  0.02508424340287435
  0.0010955002803866903

In [15]:
NNF.jacobian(a_end)

6×6 StaticArraysCore.SMatrix{6, 6, Float64, 36} with indices SOneTo(6)×SOneTo(6):
  0.691096      0.33439       0.000609166  …  -0.00021779   0.00484671
 -0.498014      1.20303      -0.001456         0.00580697   0.00109282
 -0.000155324  -0.000291939   0.139188         4.63464e-7   1.80551e-6
 -0.00129553   -0.00170724   -3.36416          3.16731e-6  -6.26364e-5
 -0.00640813    0.0043149    -1.39552e-5       2.77704     -0.791761
 -0.000705994   0.000194329  -9.7348e-9    …   0.0829852    0.334915

In [89]:
scalar(compute_sagan_rubin(factorize(a).a1∘canonize(factorize(a).a1,damping=true)).beta[1])

0.590660168880395

In [86]:
scalar(compute_sagan_rubin(a1∘r_c).beta[1])

0.590660168880395

In [91]:
b0.coords.v .= view((a∘r_c).v, 1:6)'
#b0 = Bunch((co+a.v)');
track!(b0, ring);
aend = DAMap(v=b0.coords.v)
dampend = zeros(3)
rend = canonize(aend, damping=true, damp=dampend);
a0e, a1e, a2e = factorize(aend∘rend)
scalar(compute_sagan_rubin(factorize(aend).a1∘canonize(factorize(aend).a1,damping=true)).beta[1])

0.5866968053123066

In [77]:
phase = [0.,0.,0.]
rend = canonize(aend, phase=phase, damping=true, damp=dampend);
dampend

3-element Vector{Float64}:
 0.0008124780226362984
 0.0010549097317526213
 0.002351194915378987

In [78]:
phase

3-element Vector{Float64}:
  0.0700700373959855
  0.15344937330719371
 -0.04382444058656845

In [74]:
NNF.jacobian(ci_map(a)*inv(a)*m*a*c_map(a))

6×6 StaticArraysCore.SMatrix{6, 6, ComplexF64, 36} with indices SOneTo(6)×SOneTo(6):
     0.899224-0.435093im     …   5.04549e-18+1.43746e-18im
  9.99201e-16+1.94289e-16im     -1.38955e-19+2.81743e-18im
  1.06005e-16-1.30363e-17im      1.94005e-18-4.95366e-17im
 -6.81705e-17-7.37268e-17im      6.83323e-17+5.58629e-17im
 -2.37277e-17+5.2132e-18im      -1.66533e-16-3.05311e-16im
 -2.93242e-17+6.59316e-18im  …      0.959645-0.273604im

In [76]:
 imag(log(0.899224-0.435093im ))/(2*pi*im)

-0.0 + 0.0717227709692705im